# Kolmogorov-Smirnov Test vs Wasserstein Distance for Drift Detection

## Learning Objectives

In this notebook, you will learn:
- How the Kolmogorov-Smirnov (KS) test works for drift detection
- How the Wasserstein distance measures distribution differences
- When to use KS test vs Wasserstein distance
- How to interpret the results of both methods
- Comparative analysis on synthetic drift scenarios

## Introduction

### Kolmogorov-Smirnov (KS) Test

The KS test is a non-parametric statistical test that determines whether two samples come from the same continuous distribution. It compares the cumulative distribution functions (CDFs) of the two samples and calculates the maximum vertical distance between them.

**Key characteristics:**
- Non-parametric (no assumptions about distribution)
- Provides statistical significance (p-value)
- Sensitive to differences in both location and shape
- Can be overly sensitive with large sample sizes

### Wasserstein Distance

The Wasserstein distance, also known as the Earth Mover's Distance (EMD), measures the minimum "work" required to transform one distribution into another. It considers the magnitude of differences between distributions.

**Key characteristics:**
- Considers magnitude of differences
- Stable and interpretable
- Not affected by sample size
- No built-in statistical significance test

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wasserstein_distance

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Generate Synthetic Drift Scenarios

We'll create different types of drift to compare how KS test and Wasserstein distance respond.

In [ ]:
# Sample size
n_samples = 1000

# Scenario 1: No drift (same distribution)
ref_no_drift = np.random.normal(loc=0, scale=1, size=n_samples)
mon_no_drift = np.random.normal(loc=0, scale=1, size=n_samples)

# Scenario 2: Location shift (mean change)
ref_location = np.random.normal(loc=0, scale=1, size=n_samples)
mon_location = np.random.normal(loc=0.5, scale=1, size=n_samples)

# Scenario 3: Scale shift (variance change)
ref_scale = np.random.normal(loc=0, scale=1, size=n_samples)
mon_scale = np.random.normal(loc=0, scale=1.5, size=n_samples)

# Scenario 4: Shape shift (distribution type change)
ref_shape = np.random.normal(loc=0, scale=1, size=n_samples)
mon_shape = np.random.exponential(scale=1, size=n_samples) - 1

# Scenario 5: Bimodal shift
ref_bimodal = np.random.normal(loc=0, scale=1, size=n_samples)
mon_bimodal = np.concatenate([
    np.random.normal(loc=-1, scale=0.5, size=n_samples//2),
    np.random.normal(loc=1, scale=0.5, size=n_samples//2)
])

# Scenario 6: Small shift (subtle drift)
ref_small = np.random.normal(loc=0, scale=1, size=n_samples)
mon_small = np.random.normal(loc=0.1, scale=1, size=n_samples)

print("Drift scenarios created successfully!")

## 2. Implement Drift Detection Functions

In [ ]:
def detect_drift_ks(reference, monitored):
    """
    Detect drift using Kolmogorov-Smirnov test.
    
    Returns:
    --------
    statistic : float
        KS statistic (maximum distance between CDFs)
    p_value : float
        Statistical significance
    drift_detected : bool
        True if p-value < 0.05
    """
    statistic, p_value = stats.ks_2samp(reference, monitored)
    drift_detected = p_value < 0.05
    return statistic, p_value, drift_detected

def detect_drift_wasserstein(reference, monitored):
    """
    Detect drift using Wasserstein distance.
    
    Returns:
    --------
    distance : float
        Wasserstein distance (Earth Mover's Distance)
    """
    distance = wasserstein_distance(reference, monitored)
    return distance

def visualize_drift_comparison(reference, monitored, title):
    """
    Visualize distributions and CDFs for drift comparison.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Histograms
    axes[0].hist(reference, bins=30, alpha=0.5, label='Reference', density=True, color='blue')
    axes[0].hist(monitored, bins=30, alpha=0.5, label='Monitored', density=True, color='red')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Density')
    axes[0].set_title(f'{title}\nDistribution Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: CDFs
    ref_sorted = np.sort(reference)
    mon_sorted = np.sort(monitored)
    ref_cdf = np.arange(1, len(ref_sorted) + 1) / len(ref_sorted)
    mon_cdf = np.arange(1, len(mon_sorted) + 1) / len(mon_sorted)
    
    axes[1].plot(ref_sorted, ref_cdf, label='Reference', color='blue', linewidth=2)
    axes[1].plot(mon_sorted, mon_cdf, label='Monitored', color='red', linewidth=2)
    axes[1].set_xlabel('Value')
    axes[1].set_ylabel('Cumulative Probability')
    axes[1].set_title('Cumulative Distribution Functions')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Metrics
    ks_stat, ks_pval, ks_drift = detect_drift_ks(reference, monitored)
    wass_dist = detect_drift_wasserstein(reference, monitored)
    
    metrics_text = f"KS Statistic: {ks_stat:.4f}\n"
    metrics_text += f"KS p-value: {ks_pval:.4f}\n"
    metrics_text += f"Drift Detected (KS): {'Yes' if ks_drift else 'No'}\n\n"
    metrics_text += f"Wasserstein Distance: {wass_dist:.4f}"
    
    axes[2].text(0.1, 0.5, metrics_text, fontsize=14, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[2].axis('off')
    axes[2].set_title('Drift Detection Metrics')
    
    plt.tight_layout()
    plt.show()
    
    return ks_stat, ks_pval, wass_dist

## 3. Compare Methods Across Different Drift Scenarios

In [ ]:
# Scenario 1: No drift
print("=" * 60)
print("SCENARIO 1: No Drift")
print("=" * 60)
ks1, ksp1, w1 = visualize_drift_comparison(ref_no_drift, mon_no_drift, "No Drift")

In [ ]:
# Scenario 2: Location shift
print("=" * 60)
print("SCENARIO 2: Location Shift (Mean Change)")
print("=" * 60)
ks2, ksp2, w2 = visualize_drift_comparison(ref_location, mon_location, "Location Shift")

In [ ]:
# Scenario 3: Scale shift
print("=" * 60)
print("SCENARIO 3: Scale Shift (Variance Change)")
print("=" * 60)
ks3, ksp3, w3 = visualize_drift_comparison(ref_scale, mon_scale, "Scale Shift")

In [ ]:
# Scenario 4: Shape shift
print("=" * 60)
print("SCENARIO 4: Shape Shift (Distribution Type Change)")
print("=" * 60)
ks4, ksp4, w4 = visualize_drift_comparison(ref_shape, mon_shape, "Shape Shift")

In [ ]:
# Scenario 5: Bimodal shift
print("=" * 60)
print("SCENARIO 5: Bimodal Shift")
print("=" * 60)
ks5, ksp5, w5 = visualize_drift_comparison(ref_bimodal, mon_bimodal, "Bimodal Shift")

In [ ]:
# Scenario 6: Small shift
print("=" * 60)
print("SCENARIO 6: Small Shift (Subtle Drift)")
print("=" * 60)
ks6, ksp6, w6 = visualize_drift_comparison(ref_small, mon_small, "Small Shift")

## 4. Summary Comparison Table

In [ ]:
# Create summary table
summary_df = pd.DataFrame({
    'Scenario': ['No Drift', 'Location Shift', 'Scale Shift', 'Shape Shift', 'Bimodal Shift', 'Small Shift'],
    'KS Statistic': [ks1, ks2, ks3, ks4, ks5, ks6],
    'KS p-value': [ksp1, ksp2, ksp3, ksp4, ksp5, ksp6],
    'KS Drift Detected': [ksp1 < 0.05, ksp2 < 0.05, ksp3 < 0.05, ksp4 < 0.05, ksp5 < 0.05, ksp6 < 0.05],
    'Wasserstein Distance': [w1, w2, w3, w4, w5, w6]
})

print("\nSummary of Drift Detection Results:")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

## 5. Sensitivity Analysis: Sample Size Effect

In [ ]:
# Test how sample size affects KS test sensitivity
sample_sizes = [100, 500, 1000, 2000, 5000, 10000]
ks_stats = []
ks_pvals = []
wass_dists = []

# Use small shift scenario
for n in sample_sizes:
    ref = np.random.normal(loc=0, scale=1, size=n)
    mon = np.random.normal(loc=0.1, scale=1, size=n)
    
    ks_stat, ks_pval, _ = detect_drift_ks(ref, mon)
    wass_dist = detect_drift_wasserstein(ref, mon)
    
    ks_stats.append(ks_stat)
    ks_pvals.append(ks_pval)
    wass_dists.append(wass_dist)

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# KS Statistic vs Sample Size
axes[0].plot(sample_sizes, ks_stats, marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Sample Size')
axes[0].set_ylabel('KS Statistic')
axes[0].set_title('KS Statistic vs Sample Size')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# KS p-value vs Sample Size
axes[1].plot(sample_sizes, ks_pvals, marker='o', linewidth=2, markersize=8, color='orange')
axes[1].axhline(y=0.05, color='red', linestyle='--', label='Significance Level (0.05)')
axes[1].set_xlabel('Sample Size')
axes[1].set_ylabel('KS p-value')
axes[1].set_title('KS p-value vs Sample Size')
axes[1].set_yscale('log')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Wasserstein Distance vs Sample Size
axes[2].plot(sample_sizes, wass_dists, marker='o', linewidth=2, markersize=8, color='green')
axes[2].set_xlabel('Sample Size')
axes[2].set_ylabel('Wasserstein Distance')
axes[2].set_title('Wasserstein Distance vs Sample Size')
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log')

plt.tight_layout()
plt.show()

print("\nObservation: KS test becomes more sensitive with larger sample sizes,")
print("while Wasserstein distance remains relatively stable.")

## 6. Practical Example: E-commerce Product Prices

In [ ]:
# Simulate product price data
# Reference: Normal pricing period
prices_reference = np.random.lognormal(mean=3.5, sigma=0.5, size=2000)

# Monitored: Sale period (prices shifted down)
prices_monitored = np.random.lognormal(mean=3.3, sigma=0.5, size=2000)

print("E-commerce Product Prices Analysis")
print("=" * 60)
print(f"Reference Period - Mean: ${prices_reference.mean():.2f}, Median: ${np.median(prices_reference):.2f}")
print(f"Monitored Period - Mean: ${prices_monitored.mean():.2f}, Median: ${np.median(prices_monitored):.2f}")
print("\n")

visualize_drift_comparison(prices_reference, prices_monitored, "Product Prices: Normal vs Sale Period")

# Interpretation
ks_stat, ks_pval, ks_drift = detect_drift_ks(prices_reference, prices_monitored)
wass_dist = detect_drift_wasserstein(prices_reference, prices_monitored)

print("\nInterpretation:")
print("-" * 60)
if ks_drift:
    print("✓ KS test detected significant drift (p < 0.05)")
    print("  → The price distribution has changed significantly")
    print("  → Recommendation: Retrain pricing model or update business rules")
else:
    print("✗ KS test did not detect significant drift")

print(f"\nWasserstein distance: {wass_dist:.4f}")
print(f"  → Average price shift magnitude: ${wass_dist:.2f}")
print("  → This represents the 'effort' to transform one distribution to another")

## Key Takeaways

### When to Use KS Test

1. **Need statistical significance**: When you need a p-value to make decisions
2. **Binary decision**: When you need a yes/no answer about drift
3. **Continuous data**: KS test is designed for continuous distributions
4. **Moderate sample sizes**: Works well with moderate sample sizes (100-10,000)

### When to Use Wasserstein Distance

1. **Magnitude matters**: When you need to know how much drift occurred
2. **Stability across sample sizes**: When sample sizes vary significantly
3. **Interpretability**: When you need an interpretable distance metric
4. **Ranking drift severity**: When comparing drift across multiple features

### Comparison Summary

| Aspect | KS Test | Wasserstein Distance |
|--------|---------|---------------------|
| Output | Statistic + p-value | Distance value |
| Interpretation | Statistical significance | Magnitude of shift |
| Sample size sensitivity | High (more sensitive with larger samples) | Low (stable) |
| Computational cost | Low | Moderate |
| Best for | Binary drift detection | Quantifying drift magnitude |

### Practical Recommendations

1. **Use both metrics together** for comprehensive drift monitoring
2. **KS test for alerting**: Use p-value < 0.05 as a trigger for investigation
3. **Wasserstein for prioritization**: Use distance to rank features by drift severity
4. **Consider sample size**: Be cautious with KS test on very large samples (may detect trivial differences)
5. **Business context**: Always interpret drift metrics in the context of business impact

## Exercises

1. Implement a function that uses both KS test and Wasserstein distance to create a drift severity score.

2. Create a time-series simulation where drift gradually increases, and plot how both metrics evolve.

3. Compare KS test and Wasserstein distance with Jensen-Shannon divergence on the same datasets.

4. Design a multi-feature drift monitoring dashboard that uses both metrics to rank features by drift severity.